In [1]:
import pandas as pd
import os

data_dir = "data/"
files = [f for f in os.listdir(data_dir) if f.endswith(".tsv")]

for fname in sorted(files):
    df = pd.read_csv(f"{data_dir}/{fname}", sep="\t", comment="#", low_memory=False)
    print(f"\n{'='*60}")
    print(f"FILE: {fname}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Null counts:\n{df.isnull().sum()}")
    print(df.head(3))


FILE: GeneProductAllIdentifiersSet.tsv
Shape: (4759, 11)
Columns: ['1)geneId', '2)geneName', '3)leftEndPos', '4)rightEndPos', '5)strand', '6)geneSynonyms', '7)otherDbsGeneIds', '8)productId', '9)productName', '10)productSynonyms', '11)otherDbsProductsIds ']
Null counts:
1)geneId                     0
2)geneName                   0
3)leftEndPos                 3
4)rightEndPos                3
5)strand                     0
6)geneSynonyms               0
7)otherDbsGeneIds            0
8)productId                 95
9)productName               95
10)productSynonyms         195
11)otherDbsProductsIds       1
dtype: int64
           1)geneId 2)geneName  3)leftEndPos  4)rightEndPos 5)strand  \
0  RDBECOLIGNC00001        alr     4265782.0      4266861.0  forward   
1  RDBECOLIGNC00002       modB      795862.0       796551.0  forward   
2  RDBECOLIGNC00003       cysZ     2531463.0      2532224.0  forward   

      6)geneSynonyms                                  7)otherDbsGeneIds  \
0       al

In [2]:
promoters  = pd.read_csv("data/PromoterSet.tsv",               sep="\t", comment="#")
tfs        = pd.read_csv("data/TFSet.tsv",                     sep="\t", comment="#")
genes      = pd.read_csv("data/GeneProductSet.tsv",            sep="\t", comment="#")
sigma_gene = pd.read_csv("data/NetworkSigmaGene.tsv",          sep="\t", comment="#")
tf_ri      = pd.read_csv("data/TF-RISet.tsv",                  sep="\t", comment="#")
tu_prom    = pd.read_csv("data/TuPromOperonTFSet.tsv",         sep="\t", comment="#")
id_map     = pd.read_csv("data/GeneProductAllIdentifiersSet.tsv", sep="\t", comment="#")

promoters.columns  = [c.split(')')[1] for c in promoters.columns]
tfs.columns        = [c.split(')')[1] for c in tfs.columns]
genes.columns      = [c.split(')')[1] for c in genes.columns]
sigma_gene.columns = [c.split(')')[1] for c in sigma_gene.columns]
tf_ri.columns      = [c.split(')')[1] for c in tf_ri.columns]
tu_prom.columns    = [c.split(')')[1] for c in tu_prom.columns]
id_map.columns     = [c.split(')')[1] for c in id_map.columns]

# Node tables
# 1. Gene nodes
node_genes = genes[['geneId','geneName','bnumber',
                     'leftEndPos','rightEndPos','strand']].drop_duplicates()
print(f"Gene nodes:     {len(node_genes)}")

# 2. Promoter nodes
node_promoters = promoters[['pmId','pmName','strand','posTSS',
                             'sigmaFactor','pmSequence']].dropna(subset=['pmSequence'])
print(f"Promoter nodes: {len(node_promoters)}")

# 3. TF nodes
node_tfs = tfs[['tfId','tfName','geneCodingForTF',
                 'geneBnumberCodingForTF']].drop_duplicates()
print(f"TF nodes:       {len(node_tfs)}")

# 4. Sigma factor nodes
node_sigma = pd.DataFrame(
    promoters['sigmaFactor'].dropna().unique(), columns=['sigmaName']
)
print(f"Sigma nodes:    {len(node_sigma)}")

# Edge tables

name_to_id = genes.set_index('geneName')['geneId'].to_dict()

# Edge 1: SigmaFactor --BINDS--> Promoter
edge_sigma_promoter = tu_prom[['sigmaFactor','promoterId']].dropna()
edge_sigma_promoter = edge_sigma_promoter.rename(
    columns={'sigmaFactor':'src','promoterId':'dst'})
edge_sigma_promoter['edge_type'] = 'BINDS'
print(f"Sigma→Promoter edges: {len(edge_sigma_promoter)}")

# Edge 2: TF --REGULATES--> Gene
def extract_gene_names(val):
    if pd.isna(val): return []
    parts = val.split(':')
    if len(parts) < 2: return []
    return [g.strip() for g in parts[1].split('-')]

tf_ri_exp = tf_ri[['regulatorId','targetTuOrGene']].dropna()
rows = []
for _, row in tf_ri_exp.iterrows():
    for gname in extract_gene_names(row['targetTuOrGene']):
        gid = name_to_id.get(gname)
        if gid:
            rows.append({'src': row['regulatorId'], 'dst': gid,
                         'edge_type': 'REGULATES'})
edge_tf_gene = pd.DataFrame(rows).drop_duplicates()
print(f"TF→Gene edges:        {len(edge_tf_gene)}")

# Edge 3: Promoter --DRIVES--> Gene
tu_has_prom = tu_prom.dropna(subset=['promoterId'])
rows = []
for _, row in tu_has_prom.iterrows():
    for gid in str(row['tuGenesIds']).split(','):
        gid = gid.strip()
        if gid.startswith('RDBECOLIGNC'):
            rows.append({'src': row['promoterId'], 'dst': gid,
                         'edge_type': 'DRIVES'})
edge_prom_gene = pd.DataFrame(rows).drop_duplicates()
print(f"Promoter→Gene edges:  {len(edge_prom_gene)}")

# Edge 4: TF --COOPERATES_WITH--> TF
rows = []
for _, row in tu_prom.dropna(subset=['tfsIds']).iterrows():
    tf_ids = [t.strip() for t in str(row['tfsIds']).split(',')]
    for i in range(len(tf_ids)):
        for j in range(i+1, len(tf_ids)):
            rows.append({'src': tf_ids[i], 'dst': tf_ids[j],
                         'edge_type': 'COOPERATES_WITH'})
edge_tf_tf = pd.DataFrame(rows).drop_duplicates()
print(f"TF↔TF edges:          {len(edge_tf_tf)}")

print("\n=== GRAPH SUMMARY ===")
print(f"Nodes: {len(node_genes) + len(node_promoters) + len(node_tfs) + len(node_sigma)}")
print(f"Edges: {len(edge_sigma_promoter) + len(edge_tf_gene) + len(edge_prom_gene) + len(edge_tf_tf)}")

Gene nodes:     4748
Promoter nodes: 3964
TF nodes:       243
Sigma nodes:    7
Sigma→Promoter edges: 1636
TF→Gene edges:        3223
Promoter→Gene edges:  4574
TF↔TF edges:          1734

=== GRAPH SUMMARY ===
Nodes: 8962
Edges: 11167


In [3]:
os.makedirs("data/processed", exist_ok=True)

# Node tables
node_genes.to_csv("data/processed/nodes_genes.csv", index=False)
node_promoters.to_csv("data/processed/nodes_promoters.csv", index=False)
node_tfs.to_csv("data/processed/nodes_tfs.csv", index=False)
node_sigma.to_csv("data/processed/nodes_sigma.csv", index=False)

# Edge tables
edge_sigma_promoter.to_csv("data/processed/edges_sigma_binds_promoter.csv", index=False)
edge_tf_gene.to_csv("data/processed/edges_tf_regulates_gene.csv", index=False)
edge_prom_gene.to_csv("data/processed/edges_promoter_drives_gene.csv", index=False)
edge_tf_tf.to_csv("data/processed/edges_tf_cooperates_tf.csv", index=False)

In [4]:
seq_lengths = node_promoters['pmSequence'].str.len()
print("=== Promoter Sequence Length Distribution ===")
print(seq_lengths.describe())
print(f"\nUnique lengths: {seq_lengths.nunique()}")
print("\nTop 10 most common lengths:")
print(seq_lengths.value_counts().head(10))

=== Promoter Sequence Length Distribution ===
count    3964.0
mean       81.0
std         0.0
min        81.0
25%        81.0
50%        81.0
75%        81.0
max        81.0
Name: pmSequence, dtype: float64

Unique lengths: 1

Top 10 most common lengths:
pmSequence
81    3964
Name: count, dtype: int64


In [5]:
import numpy as np

SEQ_LEN = 81
NUCLEOTIDES = {'a': 0, 't': 1, 'g': 2, 'c': 3}

def one_hot_encode(seq, length=SEQ_LEN):
    """Encode a DNA sequence to a (4, length) one-hot matrix.
       Pads with zeros if shorter, truncates if longer."""
    seq = seq.lower()[:length]
    matrix = np.zeros((4, length), dtype=np.float32)
    for i, nuc in enumerate(seq):
        if nuc in NUCLEOTIDES:
            matrix[NUCLEOTIDES[nuc], i] = 1.0
    return matrix

sequences_encoded = np.stack(
    node_promoters['pmSequence'].apply(one_hot_encode).values
)

print(f"Encoded sequence matrix shape: {sequences_encoded.shape}")
print(f"Expected: ({len(node_promoters)}, 4, {SEQ_LEN})")

np.save("data/processed/promoter_sequences_onehot.npy", sequences_encoded)
print("Saved to data/processed/promoter_sequences_onehot.npy")

Encoded sequence matrix shape: (3964, 4, 81)
Expected: (3964, 4, 81)
Saved to data/processed/promoter_sequences_onehot.npy


In [6]:
gene_id_map     = {gid: i for i, gid in enumerate(node_genes['geneId'])}
promoter_id_map = {pid: i for i, pid in enumerate(node_promoters['pmId'])}
tf_id_map       = {tid: i for i, tid in enumerate(node_tfs['tfId'])}
sigma_id_map    = {sid: i for i, sid in enumerate(node_sigma['sigmaName'])}

print(f"Gene ID space:     0 – {len(gene_id_map)-1}")
print(f"Promoter ID space: 0 – {len(promoter_id_map)-1}")
print(f"TF ID space:       0 – {len(tf_id_map)-1}")
print(f"Sigma ID space:    0 – {len(sigma_id_map)-1}")

import json
with open("data/processed/id_maps.json", "w") as f:
    json.dump({
        "gene":     gene_id_map,
        "promoter": promoter_id_map,
        "tf":       tf_id_map,
        "sigma":    sigma_id_map
    }, f)
print("\nID maps saved to data/processed/id_maps.json")

Gene ID space:     0 – 4747
Promoter ID space: 0 – 3963
TF ID space:       0 – 242
Sigma ID space:    0 – 6

ID maps saved to data/processed/id_maps.json


In [7]:
import torch

def build_edge_index(df, src_col, dst_col, src_map, dst_map):
    """Convert a string-ID edge DataFrame to a PyG edge_index tensor.
       Drops edges where either endpoint is not in the ID map."""
    src = df[src_col].map(src_map)
    dst = df[dst_col].map(dst_map)
    mask = src.notna() & dst.notna()
    dropped = (~mask).sum()
    if dropped > 0:
        print(f"  Dropped {dropped} edges with unresolved IDs")
    src = torch.tensor(src[mask].values.astype(int), dtype=torch.long)
    dst = torch.tensor(dst[mask].values.astype(int), dtype=torch.long)
    return torch.stack([src, dst], dim=0)

print("Building edge index tensors...")

ei_sigma_promoter = build_edge_index(
    edge_sigma_promoter, 'src', 'dst', sigma_id_map, promoter_id_map)

ei_tf_gene = build_edge_index(
    edge_tf_gene, 'src', 'dst', tf_id_map, gene_id_map)

ei_prom_gene = build_edge_index(
    edge_prom_gene, 'src', 'dst', promoter_id_map, gene_id_map)

ei_tf_tf = build_edge_index(
    edge_tf_tf, 'src', 'dst', tf_id_map, tf_id_map)

print(f"\nSigma→Promoter edge_index shape: {ei_sigma_promoter.shape}")
print(f"TF→Gene edge_index shape:        {ei_tf_gene.shape}")
print(f"Promoter→Gene edge_index shape:  {ei_prom_gene.shape}")
print(f"TF↔TF edge_index shape:          {ei_tf_tf.shape}")

Building edge index tensors...
  ⚠ Dropped 25 edges with unresolved IDs
  ⚠ Dropped 184 edges with unresolved IDs

Sigma→Promoter edge_index shape: torch.Size([2, 1611])
TF→Gene edge_index shape:        torch.Size([2, 3223])
Promoter→Gene edge_index shape:  torch.Size([2, 4390])
TF↔TF edge_index shape:          torch.Size([2, 1734])


In [11]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.2 MB/s eta 0:00:00


In [12]:
from torch_geometric.data import HeteroData

data = HeteroData()

data['promoter'].x = torch.tensor(
    sequences_encoded.reshape(len(node_promoters), -1), dtype=torch.float
)

strand_map = {'forward': 1.0, 'reverse': -1.0}
gene_feats = node_genes[['leftEndPos','rightEndPos','strand']].copy()
gene_feats['strand'] = gene_feats['strand'].map(strand_map).fillna(0.0)
gene_feats[['leftEndPos','rightEndPos']] = gene_feats[
    ['leftEndPos','rightEndPos']].fillna(0.0)
data['gene'].x = torch.tensor(
    gene_feats.values.astype(np.float32), dtype=torch.float
)

data['tf'].x = torch.eye(len(node_tfs), dtype=torch.float)

data['sigma'].x = torch.eye(len(node_sigma), dtype=torch.float)

data['sigma',   'binds',       'promoter'].edge_index = ei_sigma_promoter
data['tf',      'regulates',   'gene'].edge_index     = ei_tf_gene
data['promoter','drives',      'gene'].edge_index     = ei_prom_gene
data['tf',      'cooperates',  'tf'].edge_index       = ei_tf_tf

print(data)
print(f"\nNode feature shapes:")
for ntype in data.node_types:
    print(f"  {ntype}: {data[ntype].x.shape}")

HeteroData(
  promoter={ x=[3964, 324] },
  gene={ x=[4748, 3] },
  tf={ x=[243, 243] },
  sigma={ x=[7, 7] },
  (sigma, binds, promoter)={ edge_index=[2, 1611] },
  (tf, regulates, gene)={ edge_index=[2, 3223] },
  (promoter, drives, gene)={ edge_index=[2, 4390] },
  (tf, cooperates, tf)={ edge_index=[2, 1734] }
)

Node feature shapes:
  promoter: torch.Size([3964, 324])
  gene: torch.Size([4748, 3])
  tf: torch.Size([243, 243])
  sigma: torch.Size([7, 7])


In [14]:
torch.save(data, "data/processed/ecoli_hetero_graph.pt")
print("Heterogeneous graph saved to data/processed/ecoli_hetero_graph.pt")
print("\nTo reload in any future notebook:")
print("  data = torch.load('data/processed/ecoli_hetero_graph.pt')")

# Final summary
print(f"\n=== FINAL GRAPH SUMMARY ===")
print(f"Node types: {data.node_types}")
print(f"Edge types: {data.edge_types}")
total_nodes = sum(data[n].x.shape[0] for n in data.node_types)
total_edges = sum(data[e].edge_index.shape[1] for e in data.edge_types)
print(f"Total nodes: {total_nodes}")
print(f"Total edges: {total_edges}")

Heterogeneous graph saved to data/processed/ecoli_hetero_graph.pt

To reload in any future notebook:
  data = torch.load('data/processed/ecoli_hetero_graph.pt')

=== FINAL GRAPH SUMMARY ===
Node types: ['promoter', 'gene', 'tf', 'sigma']
Edge types: [('sigma', 'binds', 'promoter'), ('tf', 'regulates', 'gene'), ('promoter', 'drives', 'gene'), ('tf', 'cooperates', 'tf')]
Total nodes: 8962
Total edges: 10958
